# GameTheory-02d-Epistemique-Connaissance-Commune-Python

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) | [2c - Dilemme du voyageur](GameTheory-02c-Travelers-Dilemma-Python.ipynb)

> **Série** : GameTheory-02d-Epistemique-Connaissance-Commune-Python — accrétion du palier 02 (forme normale, dominance, élimination itérée). Le palier 02 élimine les stratégies dominées et énumère les supports mixtes sans jamais dire **ce qu'un joueur sait** ; ce carnet construit ce socle : modèles à partitions et opérateur de connaissance (Bonanno, ch. 7), connaissance commune comme point fixe, puis le théorème d'Aumann (1976) vérifié par énumération exhaustive sur un monde à six états.

## 1. Le trou que ce carnet comble

Le palier 02 pose l'élimination itérée des stratégies dominées comme une **procédure** : on efface les dominées, puis les dominées dans le jeu réduit, et ainsi de suite. La procédure est correcte, mais elle postule sa propre légitimité — *pourquoi* un joueur devrait-il savoir que l'autre va écarter ses dominées ? La réponse exige de représenter **ce que chaque joueur sait** de l'état du monde et de la connaissance de l'autre : c'est le programme épistémique de Bonanno (ch. 7-9), dont le palier principal de la série ne montre que le résultat final.

Ce carnet en exécute les trois étages, chacun avec un résultat **falsifiable** — mesuré, pas affirmé :

1. **L'opérateur de connaissance** sur un monde fini à partitions, et la vérification par énumération exhaustive des propriétés S5 (Bonanno, exercice 7.5).
2. **La connaissance commune comme point fixe** de l'itération des opérateurs individuels — avec un témoin mesuré où la connaissance mutuelle du second ordre tient sans que la connaissance commune tienne.
3. **Le théorème d'Aumann (1976)** : deux agents de priors communs dont les posteriors sont connaissance commune ne peuvent pas les savoir différents — vérifié par balayage exhaustif de tous les couples de partitions d'un monde à six états, avec contrôle négatif à priors distincts.

Les définitions et notations suivent la copie archivée du manuel (Bonanno, *Game Theory*, 2015, ch. 7-9) ; le théorème est cité depuis la copie archivée de l'article (Aumann, « Agreeing to Disagree », *Annals of Statistics* 4(6), 1976). Les deux références sont au gisement bibliographique du projet (voir la bibliographie en fin de carnet).

### 1.1 Le modèle à partitions (Bonanno, définitions 7.1-7.3)

Un **monde** est un ensemble fini d'états $W = \{w_0, w_1, \dots\}$ : chaque état décrit complètement une situation. Un **modèle d'information** attache à chaque joueur $i$ une **partition** $\mathcal{I}_i$ de $W$ ; la cellule $\mathcal{I}_i(w)$ est l'ensemble des états que le joueur $i$ ne peut pas distinguer de $w$ au vu de son information.

Un **événement** est une partie $E \subseteq W$. La définition 7.2 de Bonanno : le joueur $i$ **connaît** $E$ en $w$ si et seulement si $\mathcal{I}_i(w) \subseteq E$ — il connaît $E$ quand tous les états qu'il considère possibles réalisent $E$. La définition 7.3 en déduit l'**opérateur de connaissance** :

$$K_i(E) \;=\; \{\, w \in W : \mathcal{I}_i(w) \subseteq E \,\}$$

Le monde d'exemple ci-dessous servira tout le carnet : quatre états, deux joueurs dont les partitions se croisent.

In [1]:
W = ('w0', 'w1', 'w2', 'w3')

# Partitions en tuples de frozensets (Bonanno, definition 7.1)
I1 = (frozenset({'w0', 'w1'}), frozenset({'w2', 'w3'}))
I2 = (frozenset({'w1'}), frozenset({'w0', 'w2'}), frozenset({'w3'}))

PARTITIONS = {'joueur 1': I1, 'joueur 2': I2}

def cell_of(partition, w):
    """Cellule d'information de l'etat w pour cette partition."""
    return next(c for c in partition if w in c)

def K(partition, event):
    """Operateur de connaissance (Bonanno, definition 7.3) :
    les etats dont la cellule d'information est contenue dans l'evenement."""
    return frozenset(w for w in W if cell_of(partition, w) <= event)

def fmt(ev):
    return '{' + ', '.join(w for w in W if w in ev) + '}' if ev else '{} (vide)'

E = frozenset({'w0', 'w1', 'w2'})

print('Monde  W =', fmt(frozenset(W)))
for nom, part in PARTITIONS.items():
    print(f'{nom} : partition ' + ' | '.join(fmt(c) for c in part))
print()
print('Evenement E =', fmt(E))
for nom, part in PARTITIONS.items():
    print(f'K_{nom}(E) =', fmt(K(part, E)))

Monde  W = {w0, w1, w2, w3}
joueur 1 : partition {w0, w1} | {w2, w3}
joueur 2 : partition {w1} | {w0, w2} | {w3}

Evenement E = {w0, w1, w2}
K_joueur 1(E) = {w0, w1}
K_joueur 2(E) = {w0, w1, w2}


**Lecture de la sortie.** Le joueur 1 connaît $E$ exactement en $\{w_0, w_1\}$ : sa cellule $\{w_0, w_1\}$ est tout entière dans $E$, mais sa cellule $\{w_2, w_3\}$ contient $w_3 \notin E$. Le joueur 2, dont la cellule $\{w_0, w_2\}$ est tout entière dans $E$, connaît $E$ en $w_0$ comme en $w_2$ — et en $w_1$ (cellule singleton $\{w_1\} \subseteq E$). En $w_3$ en revanche, aucun des deux ne connaît $E$. L'opérateur $K$ transforme un événement en un autre événement : « le joueur $i$ connaît $E$ » est lui-même quelque chose qui est vrai ou faux à chaque état.

### 1.2 Résultat 1 — les propriétés S5, vérifiées par énumération exhaustive

L'exercice 7.5 de Bonanno énumère les propriétés de l'opérateur de connaissance d'une partition : **Véracité** ($K_iE \subseteq E$ : ce qui est connu est vrai), **Cohérence** ($K_iE \cap K_i\neg E = \emptyset$ : on ne connaît jamais une chose et son contraire), **Introspection positive** ($K_iE \subseteq K_iK_iE$ : si l'on connaît, on connaît que l'on connaît), **Introspection négative** ($\neg K_iE \subseteq K_i\neg K_iE$ : si l'on ne connaît pas, on connaît que l'on ne connaît pas), **Monotonie** ($E \subseteq F \Rightarrow K_iE \subseteq K_iF$) et **Conjonction** ($K_iE \cap K_iF = K_i(E \cap F)$).

Ces six propriétés sont le système **S5**. Plutôt que de les prouver, ce carnet les **mesure** : sur le monde à quatre états, l'ensemble $2^W$ des événements est fini — $2^4 = 16$ événements — et chaque propriété se vérifie sur chacun d'eux, pour chaque joueur, en comparant des ensembles. Un échec afficherait une ligne de violation ; la sortie ne doit montrer que des compteurs à zéro.

In [2]:
from itertools import combinations

EVENTS = [frozenset(comb)
          for r in range(len(W) + 1)
          for comb in combinations(W, r)]

def neg(event):
    """Complement d'un evenement dans W."""
    return frozenset(w for w in W if w not in event)

def violations_s5(partition):
    """Enumere tous les evenements et compte les violations de chaque propriete S5
    (Bonanno, exercice 7.5)."""
    compte = {'Veracite': 0, 'Coherence': 0, 'Introspection positive': 0,
              'Introspection negative': 0, 'Monotonie': 0, 'Conjonction': 0}
    for E in EVENTS:
        KE = K(partition, E)
        nKE = neg(KE)
        if not KE <= E:
            compte['Veracite'] += 1
        if KE & K(partition, neg(E)):
            compte['Coherence'] += 1
        if not KE <= K(partition, KE):
            compte['Introspection positive'] += 1
        if not nKE <= K(partition, nKE):
            compte['Introspection negative'] += 1
        for F in EVENTS:
            if E <= F and not KE <= K(partition, F):
                compte['Monotonie'] += 1
            if K(partition, E) & K(partition, F) != K(partition, E & F):
                compte['Conjonction'] += 1
    return compte

print(f'{len(EVENTS)} evenements enumeres pour chacun des {len(PARTITIONS)} joueurs')
print(f'({len(EVENTS)}^2 = {len(EVENTS)**2} paires pour Monotonie et Conjonction)\n')
total = 0
for nom, part in PARTITIONS.items():
    compte = violations_s5(part)
    total += sum(compte.values())
    resume = ' | '.join(f'{k}: {v}' for k, v in compte.items())
    print(f'{nom} : {resume}')
print(f'\nTOTAL violations S5 sur le monde : {total}')

16 evenements enumeres pour chacun des 2 joueurs
(16^2 = 256 paires pour Monotonie et Conjonction)

joueur 1 : Veracite: 0 | Coherence: 0 | Introspection positive: 0 | Introspection negative: 0 | Monotonie: 0 | Conjonction: 0
joueur 2 : Veracite: 0 | Coherence: 0 | Introspection positive: 0 | Introspection negative: 0 | Monotonie: 0 | Conjonction: 0

TOTAL violations S5 sur le monde : 0


**Lecture de la sortie.** Zéro violation, sur $16$ événements par joueur et $16^2 = 256$ paires pour les propriétés binaires. Les six propriétés tiennent **par construction** : l'opérateur $K$ d'une partition ne peut affirmer que des choses vraies (la cellule qui certifie $E$ est incluse dans $E$), ne peut certifier une contradiction, et est stable par itération (connaître sa propre cellule d'information n'apporte rien). La réciproque est tout aussi instructive : si l'« information » n'est pas une partition — cellules qui se chevauchent, ou état que l'agent exclut alors qu'il est possible — certaines de ces propriétés tombent. C'est l'objet de l'exercice 1.

### Exercice 1 — quand l'information n'est pas une partition

Une **correspondance de possibilité** quelconque associe à chaque état un ensemble d'états « considérés possibles », sans exiger que ces ensembles forment une partition : deux états peuvent se considérer mutuellement possibles sans être dans la même cellule, ou un état peut s'exclure lui-même. L'opérateur $K(P, E) = \{w : P(w) \subseteq E\}$ se définit pareillement, mais S5 ne tient plus forcément.

Construisez une correspondance $P$ sur le monde $W$ ci-dessus (par exemple $P(w_0) = \{w_0, w_1\}$, $P(w_1) = \{w_2\}$, $P(w_2) = \{w_0, w_1, w_2\}$, $P(w_3) = \{w_3\}$) et mesurez lesquelles des six propriétés S5 tombent, avec le même énumérateur que ci-dessus.

In [3]:
# Exercice a completer : S5 sur une correspondance de possibilite quelconque
#
# Etape 1 : definir P comme un dict etat -> frozenset(etats possibles).
# Etape 2 : adapter l'enumerateur de la cellule precedente pour lire P
#           (la cellule de w devient P[w]) et relancer le comptage.
# Etape 3 : afficher le tableau des violations et interpreter :
#           la Veracite tombe-t-elle ? l'Introspection positive ? lesquelles tiennent ?
#
# Indice : P(w0) = {w0, w1} et P(w1) = {w2} ne sont pas "coherents" au sens
# des partitions -- cherchez quel axiome echoue sur E = {w0}.

P_quelconque = None  # TODO etudiant
print('Exercice a completer : mesurer quelles proprietes S5 tombent sur P')

Exercice a completer : mesurer quelles proprietes S5 tombent sur P


## 2. Résultat 2 — la connaissance commune comme point fixe

La section 7.3 de Bonanno : l'événement $E$ est **connaissance commune** en $w$ si $w$ appartient à l'intersection infinie

$$CK(E) \;=\; K_1E \cap K_2E \cap K_1K_2E \cap K_2K_1E \cap K_1K_1K_2E \cap K_1K_2K_1E \cap \dots$$

Vérifier une infinité de conditions semble impossible ; Bonanno note que c'est au contraire « très facile », parce que sur un monde fini l'itération

$$X_0 = E, \qquad X_{n+1} = X_n \cap K_1(X_n) \cap K_2(X_n)$$

est décroissante et **stabilise en au plus $|W|$ pas** : le point fixe obtenu est exactement $CK(E)$. Le carnet exécute cette itération et affiche chaque pas — puis construit le **témoin** promis : un état où la connaissance mutuelle du second ordre tient ($K_1K_2E$ et $K_2K_1E$ tous deux vrais) alors que $CK(E)$ est vide.

In [4]:
def CK(partitions, event, trace=False):
    """Point fixe decreissant de l'intersection des operateurs (Bonanno 7.3).
    Retourne (CK(E), nombre de pas effectifs)."""
    X = frozenset(event)
    pas = 0
    if trace:
        print(f'  X_0 = E = {fmt(X)}')
    while True:
        Y = X
        for part in partitions:
            Y = Y & K(part, Y)
        if Y == X:
            return X, pas
        X = Y
        pas += 1
        if trace:
            print(f'  X_{pas}       = {fmt(X)}')

print('CK(E) pour E =', fmt(E), ':')
ck_E, n_pas = CK(list(PARTITIONS.values()), E, trace=True)
print(f'  -> point fixe atteint en {n_pas} pas : CK(E) =', fmt(ck_E))
print()
print('Connaissance mutuelle du second ordre en w1 :')
k1k2 = K(I1, K(I2, E))
k2k1 = K(I2, K(I1, E))
print('  K_1 K_2 (E) =', fmt(k1k2))
print('  K_2 K_1 (E) =', fmt(k2k1))
print('  w1 dans K_1K_2(E) :', 'w1' in k1k2, '| w1 dans K_2K_1(E) :', 'w1' in k2k1)
print('  w1 dans CK(E)      :', 'w1' in ck_E)

CK(E) pour E = {w0, w1, w2} :
  X_0 = E = {w0, w1, w2}
  X_1       = {w1}
  X_2       = {} (vide)
  -> point fixe atteint en 2 pas : CK(E) = {} (vide)

Connaissance mutuelle du second ordre en w1 :
  K_1 K_2 (E) = {w0, w1}
  K_2 K_1 (E) = {w1}
  w1 dans K_1K_2(E) : True | w1 dans K_2K_1(E) : True
  w1 dans CK(E)      : False


**Lecture de la sortie — le témoin mesuré.** En $w_1$, le joueur 1 sait que le joueur 2 sait $E$, **et** le joueur 2 sait que le joueur 1 sait $E$ : les deux intersections du second ordre contiennent $w_1$. Pourtant $CK(E) = \emptyset$ : la hiérarchie casse au troisième étage. La raison se lit dans les itérations affichées : $X_1 = \{w_1\}$, puis $X_2 = \emptyset$ parce que $K_1(\{w_1\}) = \emptyset$ — le joueur 1, dont la cellule est $\{w_0, w_1\}$, ne peut pas distinguer $w_0$ de $w_1$, donc ne peut pas savoir que l'événement s'est rétréci à $\{w_1\}$. Connaissance mutuelle $\neq$ connaissance commune : entre les deux, il y a toute une hiérarchie infinie, et sur un monde fini elle s'effondre en un point fixe atteignable en au plus $|W|$ pas — ici $2$.

### 2.1 L'équivalence par les cellules du « meet »

La caractérisation duale (Bonanno 7.3) : $E$ est connaissance commune en $w$ si et seulement si la **cellule du meet** des partitions qui contient $w$ — la plus petite réunion de cellules des deux joueurs qui soit à la fois une réunion de cellules de $\mathcal{I}_1$ et de cellules de $\mathcal{I}_2$ — est tout entière contenue dans $E$. Le meet se calcule par fermeture transitive : deux états sont dans la même cellule du meet si l'on peut passer de l'un à l'autre en enchaînant « être dans la même cellule de $\mathcal{I}_1$ ou de $\mathcal{I}_2$ ». Vérifions l'équivalence sur tous les événements du monde : chaque état doit satisfaire $w \in CK(E) \iff \mathrm{meet}(w) \subseteq E$, sans un seul contre-exemple.

In [5]:
def meet_cells(partitions, world=W):
    """Cellules du meet : fermeture transitive de 'meme cellule pour un joueur'."""
    parent = {w: w for w in world}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for part in partitions:
        for c in part:
            representant = next(iter(c))
            for w in c:
                parent[find(w)] = find(representant)
    cellules = {}
    for w in world:
        cellules.setdefault(find(w), set()).add(w)
    return [frozenset(c) for c in cellules.values()]

MEET = meet_cells(list(PARTITIONS.values()))
meet_of = {}
for c in MEET:
    for w in c:
        meet_of[w] = c

print('Cellules du meet :', ' | '.join(fmt(c) for c in MEET))
print()
contre_exemples = 0
for Ev in EVENTS:
    ck_Ev, _ = CK(list(PARTITIONS.values()), Ev)
    for w in W:
        par_iter = w in ck_Ev
        par_meet = meet_of[w] <= Ev
        if par_iter != par_meet:
            contre_exemples += 1
            print(f'  CONTRE-EXEMPLE : w = {w}, E = {fmt(Ev)}, iteration={par_iter}, meet={par_meet}')
print(f'Equivalence CK-iteration <-> meet verifiee sur {len(EVENTS)} evenements x {len(W)} etats '
      f'({len(EVENTS)*len(W)} couples) : {contre_exemples} contre-exemple(s)')

Cellules du meet : {w0, w1, w2, w3}

Equivalence CK-iteration <-> meet verifiee sur 16 evenements x 4 etats (64 couples) : 0 contre-exemple(s)


**Lecture de la sortie.** Zéro contre-exemple sur les $64$ couples (événement, état) : l'itération infinie de Bonanno et la caractérisation par le meet calculent le même objet. Le meet donne la version **publique** de la connaissance commune : une union de cellules du meet est un événement que chacun sait découper en cellules de sa propre partition — donc un événement que chacun connaît dès qu'il est vrai, et qui rend $E$ connaissance commune dès qu'il est inclus dans $E$ (exercice 2).

### Exercice 2 — l'événement public qui certifie la connaissance commune

La caractérisation du meet se réécrit : $E$ est connaissance commune en $w$ si et seulement s'il existe un **événement public** $F$ — une union de cellules du meet — tel que $F \subseteq E$ et $w \in F$. « Public » signifie ici : chacun connaît $F$ en tout état de $F$ (et même : $F$ est connaissance commune en tout état de $F$).

Pour l'événement $E = \{w_0, w_1\}$ du monde ci-dessus, déterminez toutes les unions de cellules du meet incluses dans $E$, et vérifiez sur chacune que $K_i(F) \supseteq F$ pour les deux joueurs — la signature d'un événement public.

In [6]:
# Exercice a completer : unions publiques incluses dans E = {w0, w1}
#
# Etape 1 : enumerer toutes les unions de cellules du meet (2**len(MEET) choix).
# Etape 2 : retenir celles incluses dans E = frozenset({'w0', 'w1'}).
# Etape 3 : pour chacune, verifier K_1(F) >= F et K_2(F) >= F (signature publique)
#           et comparer avec CK(E) obtenu par iteration.
#
# Indice : le meet du monde ci-dessus n'a qu'UNE cellule, le monde entier W --
# les seuls evenements publics sont donc vide et W. La reponse a
# "pourquoi CK({w0,w1}) est-il vide ?" se lit dans cette pauvrete : aucun
# evenement public non vide n'est inclus dans {w0, w1}.

print('Exercice a completer : unions publiques incluses dans E = {w0, w1}')

Exercice a completer : unions publiques incluses dans E = {w0, w1}


## 3. Résultat 3 — « Agreeing to disagree » (Aumann 1976), vérifié par énumération

Le théorème d'Aumann (1976), cité depuis la copie archivée de l'article :

> **Theorem.** « If two people have the same priors, and their posteriors for an event $A$ are common knowledge, then these posteriors are equal. » — deux personnes de **priors communs** dont les posteriors sur un événement sont **connaissance commune** ont des posteriors **égaux**. « In brief, people with the same priors cannot agree to disagree. »

La construction (Bonanno, §8.5) : chaque joueur $i$ part du même prior $p$ sur $W$ et forme son posterior en conditionnant sur sa cellule d'information, $P_i(E) = p(E \cap \mathcal{I}_i(w)) \, / \, p(\mathcal{I}_i(w))$. Le posterior de $i$ est constant sur chaque cellule de $\mathcal{I}_i$ — donc « le joueur $i$ attache la probabilité $q$ à $E$ » est un événement, auquel l'opérateur $K$ et le meet s'appliquent.

Le théorème se **mesure** : les posteriors sont connaissance commune en $w$ exactement quand les deux posteriors sont constants sur la cellule du meet de $w$ (chacun connaît son propre posterior partout dans sa cellule ; la connaissance commune exige la constance sur toute la cellule publique). Sur un monde à **six états**, il existe $203$ partitions (nombre de Bell $B_6 = 203$), donc $203^2 = 41\,209$ couples $(\mathcal{I}_1, \mathcal{I}_2)$, croisés avec les $2^6 = 64$ événements et les cellules du meet de chaque couple. Le balayage ci-dessous examine chaque cas où les deux posteriors sont connaissance commune et vérifie qu'ils sont égaux — une seule égalité fausse suffirait à réfuter le théorème sur ce monde.

In [7]:
from math import gcd

N = 6  # monde a six etats {0..5}, representes par des masques binaires

def bell_partitions(n):
    """Toutes les partitions de {0..n-1}, comme tuples de masques binaires."""
    def rec(i, blocs):
        if i == n:
            yield tuple(sorted(blocs))
            return
        vus = set()
        for k in range(len(blocs)):
            if blocs[k] in vus:
                continue
            vus.add(blocs[k])
            blocs[k] |= 1 << i
            yield from rec(i + 1, blocs)
            blocs[k] &= ~(1 << i)
        blocs.append(1 << i)
        yield from rec(i + 1, blocs)
        blocs.pop()
    yield from rec(0, [])

PARTITIONS6 = sorted(set(bell_partitions(N)))
print(f'Monde a {N} etats : {len(PARTITIONS6)} partitions (nombre de Bell B_{N}), '
      f'{len(PARTITIONS6)**2} couples, {2**N} evenements')

def meet6(p1, p2):
    """Cellules du meet de deux partitions a six etats (fermeture transitive)."""
    parent = list(range(N))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for part in (p1, p2):
        for mask in part:
            racine = None
            for s in range(N):
                if mask >> s & 1:
                    r = find(s)
                    if racine is None:
                        racine = r
                    else:
                        parent[r] = racine
    cellules = {}
    for s in range(N):
        cellules.setdefault(find(s), 0)
        cellules[find(s)] |= 1 << s
    return list(cellules.values())

def table_poids(prior):
    """table[E] = poids du prior sur les etats de E (prior en entiers)."""
    return [sum(prior[s] for s in range(N) if E >> s & 1) for E in range(2 ** N)]

def cle(a, b):
    """Cle exacte du rapport a/b, reduite par pgcd (comparaison exacte)."""
    g = gcd(a, b)
    return (a // g, b // g)

def balayage(prior1, prior2, exemple=False):
    """Balaye couples x cellules du meet x evenements. Pour chaque cas ou les
    deux posteriors sont constants sur la cellule du meet (connaissance commune),
    verifie l'egalite. Retourne (cas_examines, violations, un_exemple_ou_None)."""
    w1, w2 = table_poids(prior1), table_poids(prior2)
    cas = violations = 0
    premier = None
    for p1 in PARTITIONS6:
        for p2 in PARTITIONS6:
            for M in meet6(p1, p2):
                c1 = [c for c in p1 if c & M]
                c2 = [c for c in p2 if c & M]
                for E in range(2 ** N):
                    q1 = {cle(w1[E & c], w1[c]) for c in c1}
                    if len(q1) != 1:
                        continue
                    q2 = {cle(w2[E & c], w2[c]) for c in c2}
                    if len(q2) != 1:
                        continue
                    cas += 1
                    (a1, b1), = q1
                    (a2, b2), = q2
                    if a1 * b2 != a2 * b1:
                        violations += 1
                        if exemple and premier is None:
                            premier = (p1, p2, M, E, (a1, b1), (a2, b2))
    return cas, violations, premier

print('Balayage exhaustif (prior commun uniforme)...', flush=True)
cas, violations, _ = balayage([1] * N, [1] * N)
print(f'  cas ou les deux posteriors sont connaissance commune : {cas:,}')
print(f'  desaccords (posteriors ck et differents)            : {violations}')
print()
print('Theoreme d Aumann sur ce monde :',
      'VERIFIE (0 desaccord)' if violations == 0 else 'REFUTE')

Monde a 6 etats : 203 partitions (nombre de Bell B_6), 41209 couples, 64 evenements
Balayage exhaustif (prior commun uniforme)...


  cas ou les deux posteriors sont connaissance commune : 1,760,900
  desaccords (posteriors ck et differents)            : 0

Theoreme d Aumann sur ce monde : VERIFIE (0 desaccord)


**Lecture de la sortie.** Sur les $41\,209$ couples de partitions, le balayage isole plus d'un million de cas où les posteriors des deux joueurs sont connaissance commune (constants sur leur cellule du meet) — et **zéro** désaccord. Le théorème n'est pas une moyenne ou une tendance : sur ce monde, il n'existe *aucune* configuration d'information, aucun événement, aucune cellule publique qui permette à deux agents de priors communs de savoir en commun des posteriors différents. La preuve d'Aumann tient en une page ; l'énumération ne la remplace pas, elle la **confronte** — et c'est la confrontation qui est falsifiable.

### 3.1 Contrôle négatif — la frontière du théorème est le prior

L'hypothèse « same priors » n'est pas décorative. Donnons au joueur 2 un prior différent — même forme, autre pondération — et balayons à nouveau : le même critère de constance sur la cellule du meet isole des cas de posteriors connaissance commune, mais cette fois l'égalité peut céder. Le contrôle négatif est **mesuré** : la sortie doit montrer les deux visages, désaccord compté puis un cas concret affiché.

In [8]:
# Prior commun uniforme pour le joueur 1 ; prior tente pour le joueur 2
prior_j1 = [1, 1, 1, 1, 1, 1]
prior_j2 = [1, 1, 1, 1, 1, 4]

print('Balayage exhaustif (priors distincts : uniforme vs (1,1,1,1,1,4))...', flush=True)
cas_n, viol_n, exemple = balayage(prior_j1, prior_j2, exemple=True)
print(f'  cas ou les deux posteriors sont connaissance commune : {cas_n:,}')
print(f'  desaccords (posteriors ck et differents)            : {viol_n:,}')
print()

if exemple:
    p1, p2, M, E, (a1, b1), (a2, b2) = exemple
    bits = lambda m: '{' + ','.join(str(s) for s in range(N) if m >> s & 1) + '}'
    print('Un desaccord connaissance-commune, en detail :')
    print('  partition joueur 1 :', ' | '.join(bits(c) for c in p1))
    print('  partition joueur 2 :', ' | '.join(bits(c) for c in p2))
    print('  cellule du meet    :', bits(M), ' evenement E =', bits(E))
    print(f'  posterior joueur 1 : {a1}/{b1} = {a1/b1:.4f}   (prior uniforme)')
    print(f'  posterior joueur 2 : {a2}/{b2} = {a2/b2:.4f}   (prior tente)')
    print('  -> connaissance commune de posteriors DIFFERENTS : le theoreme ne s applique pas')
else:
    print('Aucun desaccord trouve : controle negatif a revoir')

Balayage exhaustif (priors distincts : uniforme vs (1,1,1,1,1,4))...


  cas ou les deux posteriors sont connaissance commune : 1,753,520
  desaccords (posteriors ck et differents)            : 53,962

Un desaccord connaissance-commune, en detail :
  partition joueur 1 : {0} | {1} | {2} | {3} | {4,5}
  partition joueur 2 : {0} | {1} | {2} | {3} | {4,5}
  cellule du meet    : {4,5}  evenement E = {4}
  posterior joueur 1 : 1/2 = 0.5000   (prior uniforme)
  posterior joueur 2 : 1/5 = 0.2000   (prior tente)
  -> connaissance commune de posteriors DIFFERENTS : le theoreme ne s applique pas


**Lecture de la sortie.** Avec des priors distincts, des dizaines de milliers de désaccords à posteriors connaissance commune apparaissent là où le prior commun en donnait zéro — la frontière du théorème est bien l'hypothèse de prior commun (« like-minded »), pas la quantité d'information. Le cas concret affiché montre le mécanisme : les deux joueurs partagent la même cellule publique, leurs posteriors y sont constants — donc connaissance commune — mais chacun conditionne avec sa propre pondération du monde, et les deux nombres diffèrent. Rien ne contraint deux agents qui ne « pensent pas pareil » avant d'observer.

### Exercice 3 — désaccord mutuellement connu, pas communément connu

Entre « posteriors différents » (banal : informations différentes) et « posteriors connaissance commune différents » (impossible à priors communs), il reste une case : les posteriors sont **mutuellement connus** — le joueur 1 connaît le posterior du joueur 2 et réciproquement — sans être connaissance commune. Bonanno (§8.5, figure 8.7) construit exactement ce cas sur un monde à quatre états avec prior commun uniforme, partitions $\mathcal{I}_1 = \{\{a,b,c\},\{d\}\}$ et $\mathcal{I}_2 = \{\{a,b\},\{c,d\}\}$, événement $E = \{b,c\}$ : les deux posteriors diffèrent et chacun connaît le posterior de l'autre, mais le meet n'a pas de posterior constant — pas de connaissance commune, pas de contradiction avec Aumann.

Reconstruisez ce cas : calculez les posteriors, les événements « $P_i(E) = q$ », vérifiez qu'ils sont mutuellement connus, puis montrez par le meet que la connaissance commune échoue.

In [9]:
# Exercice a completer : desaccord mutuellement connu (Bonanno 8.5, figure 8.7)
#
# Etape 1 : definir le monde W = {a,b,c,d}, les deux partitions ci-dessus,
#           le prior commun uniforme et E = {b,c} ; calculer P_1(E) et P_2(E)
#           par conditionnement bayesien sur chaque cellule.
# Etape 2 : former les evenements 'P_1(E) = q1' et 'P_2(E) = q2' et verifier
#           qu'en l'etat a, chaque joueur connait le posterior de l'autre
#           (operateur K de la section 1.1).
# Etape 3 : calculer la cellule du meet de a et montrer qu'au moins un des
#           posteriors y est non constant -> pas de connaissance commune ->
#           le theoreme d'Aumann ne s'applique pas.
#
# Indice : avec le prior uniforme, P_1(E) = P(E & {a,b,c}) / P({a,b,c}) = 2/3
#          et P_2(E) = P(E & {a,b}) / P({a,b}) = 1/2 -- verifiez ces valeurs,
#          ne les supposez pas.

print('Exercice a completer : le desaccord mutuellement connu de Bonanno 8.5')

Exercice a completer : le desaccord mutuellement connu de Bonanno 8.5


## 4. Ce que le palier 02 y gagne

Avec l'opérateur $K$ et la connaissance commune en main, la procédure du palier 02 cesse d'être un postulat. Le chapitre 9 de Bonanno en fait la dérivation : dans un jeu en forme normale, si la **rationalité** de chaque joueur est connaissance commune, les stratégies strictement dominées sont éliminées, puis — la rationalité étant connaissance commune — chacun sait que les autres ont éliminé les leurs, et l'itération continue : l'élimination itérée des stratégies dominées **est** le retracement de la connaissance commune de rationalité sur le jeu. Le palier 02 exécutait cette procédure ; ce carnet a construit l'objet (le point fixe $CK$) qui la justifie.

C'est aussi le socle des raffinements des paliers 10-12 : les croyances hors équilibre — que pense un joueur quand il observe ce que l'équilibre juge impossible — sont des posteriors sur des partitions d'information, exactement comme au §3 ; la différence est que la connaissance n'y est plus commune, seulement mutuelle, et le théorème d'Aumann marque précisément la frontière entre ce que cette connaissance mutuelle contraint et ce qu'elle laisse libre.

## Bibliographie

- Giacomo Bonanno, *Game Theory* (Open Access textbook with 165 solved exercises), arXiv 1512.06808, 2015 — chapitres 7 (Knowledge and common knowledge), 8 (Adding beliefs to knowledge), 9 (Common knowledge of rationality). Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\2015 - Bonanno - Game Theory (Open Access textbook with 165 solved exercises) - arXiv 1512.06808.pdf` (CC BY-NC-ND 4.0). Définitions 7.1-7.3, exercice 7.5 (propriétés S5), section 7.3 (connaissance commune), section 8.5 (agreeing to disagree).
- Robert J. Aumann, « Agreeing to Disagree », *The Annals of Statistics* 4(6), 1976, pp. 1236-1239. Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\1976 - Aumann - Agreeing to Disagree (Annals of Statistics 4(6) 1236-1239).pdf` (numérisation JSTOR, URL stable https://www.jstor.org/stable/2958591). Théorème cité verbatim p. 1236.

*Ce carnet est une annexe épistémique du palier 02 ; il ne modifie ni les carnets 02, ni 02b, ni 02c, et s'exécute en Python standard sans dépendance au-delà de la bibliothèque stdlib.*